# Case-01: RotSpring2D —— pyFEM 自訂轉角彈簧元素,孤立最小測試

**專案**: pyfem-plastic-hinge（仿 [taiwan-seismic-code-calc](https://github.com/zhixiu0223/taiwan-seismic-code-calc) 的 Case 序列 + Validation Log 格式）

**目標（Stage 1-2）**：在 pyFEM 裡寫一個最小的自訂元素 `RotSpring2D`（線性彈性
轉角彈簧，M = k·Δθ），跟 CalculiX 的 `HINGE2` 扮演同樣角色，但用 pyFEM
原生 Python 元素介面實作（不是 `*USER ELEMENT`）。

**驗證方式**：兩個重合節點，節點1固定當地面，節點2固定平移、轉角自由，
施加彎矩 M，數值解 θ 跟手算 M/k 比對。純線性問題，理論上應該是機器精度
等級的一致，不是「合理範圍內」的近似。

**尚未做的事（刻意留到後面 Case）**：Mp 降伏封頂、跟樑元素組合、
非線性疊代自我檢查。這個 notebook 只驗證「元素本身的線性行為對不對」。


In [ ]:
# ===== 0: 安裝 pyFEM（已安裝則略過，Colab 每次重跑不用重裝）=====
import os
PYFEM_DIR = "/content/PyFEM"

if not os.path.isdir(PYFEM_DIR):
    !git clone -q https://github.com/jjcremmers/PyFEM.git {PYFEM_DIR}
    %pip install -q -e {PYFEM_DIR} --break-system-packages
else:
    print(f"{PYFEM_DIR} 已存在，略過安裝")


## 1. 寫入自訂元素 RotSpring2D

pyFEM 用 `importlib.import_module(f"pyfem.elements.{modelType}")` 動態載入
元素類別（見 `pyfem/fem/ElementSet.py`），所以自訂元素必須是一個真實檔案，
放進已安裝套件的 `pyfem/elements/` 資料夾裡（這也是為什麼上一步用
`pip install -e`——可編輯安裝讓這裡寫的檔案立刻對套件生效，不用重裝）。


In [ ]:
# ===== 1: 把 RotSpring2D.py 寫進已安裝的 pyfem 套件 =====
rotspring_code = r'''
from pyfem.elements.Element import Element
from numpy import zeros


class RotSpring2D(Element):
    """2-node 2D 轉角彈簧元素 (pyFEM 自訂元素, Stage 1: 線性彈性)

    力學假設:
      - 兩個節點理論上重合
      - 只有轉角自由度 rz 之間有相對勁度 k: M = k * (rz2 - rz1)
      - 平移自由度 u, v 本元件不提供任何勁度貢獻 (Stage 3 跟樑元素組合時另外處理)
    """

    dofTypes = ['u', 'v', 'rz']

    def __init__(self, elnodes, props):
        Element.__init__(self, elnodes, props)
        self.family = "BEAM"

    def getTangentStiffness(self, elemdat):
        k = elemdat.props.k

        # state 向量排列: [u1, v1, rz1, u2, v2, rz2]
        theta1 = elemdat.state[2]
        theta2 = elemdat.state[5]
        dtheta = theta2 - theta1
        M = k * dtheta

        elemdat.fint = zeros(6)
        elemdat.fint[2] = -M
        elemdat.fint[5] = M

        elemdat.stiff = zeros((6, 6))
        elemdat.stiff[2, 2] = k
        elemdat.stiff[2, 5] = -k
        elemdat.stiff[5, 2] = -k
        elemdat.stiff[5, 5] = k

    def getInternalForce(self, elemdat):
        self.getTangentStiffness(elemdat)
'''

target = f"{PYFEM_DIR}/pyfem/elements/RotSpring2D.py"
with open(target, "w") as f:
    f.write(rotspring_code)
print(f"已寫入 {target}")


## 2. 孤立最小測試：θ = M/k

不透過 `.pro`/`.dat` 檔案，直接用 pyFEM 的低階 Python API 建模（`NodeSet`
+ `ElementSet` + `DofSpace`），跟官方 `examples/ch03/ShallowTrussFE.py`
同一種寫法——比 `.pro`/`.dat` 格式更適合放進 notebook 逐格說明。


In [ ]:
import sys
sys.path.insert(0, PYFEM_DIR)

from pyfem.util.dataStructures import Properties, GlobalData
from pyfem.fem.NodeSet import NodeSet
from pyfem.fem.ElementSet import ElementSet
from pyfem.fem.DofSpace import DofSpace
from pyfem.fem.Assembly import assembleTangentStiffness
from pyfem.models.ModelManager import ModelManager
from numpy import zeros

# ---------- 參數 ----------
k = 1.0e10          # 轉角彈簧勁度 (N-mm/rad)
M_applied = 1.0e8   # 施加彎矩 (N-mm)

# ---------- 建模 ----------
props = Properties()
props.HingeElem = Properties({'type': 'RotSpring2D', 'k': k})

nodes = NodeSet()
nodes.add(1, [0.0, 0.0])
nodes.add(2, [0.0, 0.0])   # 與節點1重合

elements = ElementSet(nodes, props)
elements.add(1, 'HingeElem', [1, 2])

dofs = DofSpace(elements)
cons = dofs.createConstrainer()

for dtype in ['u', 'v', 'rz']:
    cons.addConstraint(dofs.getForType(1, dtype), 0.0, "main")   # 節點1: 地面, 全固定
for dtype in ['u', 'v']:
    cons.addConstraint(dofs.getForType(2, dtype), 0.0, "main")   # 節點2: 固定平移, 轉角自由
cons.flush()

globdat = GlobalData(nodes, elements, dofs)
globdat.models = ModelManager(props, globdat)

# ---------- 施加彎矩並求解 ----------
a = globdat.state
fext = zeros(len(dofs))
loadDof = dofs.getForType(2, 'rz')
fext[loadDof] = M_applied

K, fint = assembleTangentStiffness(props, globdat)
da = dofs.solve(K, fext - fint)
a[:] += da[:]

K, fint = assembleTangentStiffness(props, globdat)
residual = dofs.norm(fext - fint)

theta2_numeric = a[loadDof]
theta2_hand = M_applied / k
rel_err = abs(theta2_numeric - theta2_hand) / abs(theta2_hand)

print("=== Case-01: RotSpring2D 孤立最小測試 ===")
print(f"施加彎矩 M       = {M_applied:.6e} N-mm")
print(f"彈簧勁度 k       = {k:.6e} N-mm/rad")
print(f"數值解 theta2    = {theta2_numeric:.10e} rad")
print(f"手算值 M/k       = {theta2_hand:.10e} rad")
print(f"相對誤差         = {rel_err:.3e}")
print(f"殘差(平衡自我檢查) = {residual:.3e}")

assert rel_err < 1e-8, "數值解與手算不符，RotSpring2D 實作有誤"
assert residual < 1e-6, "殘差未收斂到位，不是真正的平衡解"
print("\n✅ PASS")


## 3. 結論與下一步

實際在這個沙盒環境跑過（不是預期結果）：`相對誤差 = 0.000e+00`，
`殘差 = 0.000e+00`——純線性問題應有的精確一致。

**Validation Log**

| ID | 主題 | 比對對象 | 結果 |
|---|---|---|---|
| VL-01 | RotSpring2D 孤立線性行為 | 手算 M/k | 機器精度一致 (rel_err=0) |

**下一步（Case-02）**：把 `RotSpring2D` 跟 pyFEM 既有的 `BeamNL` 串接
（一端接彈簧、一端固定，尖端施力），驗證組合後的總撓度是否等於
「彈簧貢獻 + 梁本身撓度」的手算疊加值——對應 CalculiX 那條線的
UB+HINGE2 Stage 4 combination test。
